# Frank-Wolfe Optimization for the Minimum Enclosing Ball Problem

This notebook reproduces the experiments from the accompanying report: three
Frank-Wolfe variants (Standard, Pairwise, Fully-Corrective) solving the dual
formulation of the Minimum Enclosing Ball (MEB) problem on MNIST,
Fashion-MNIST and CIFAR-10, followed by a geometric anomaly detection
application built on top of the resulting balls.

Full derivations and discussion are in `report.pdf`; this notebook focuses on
running the pipeline end to end. All the algorithmic logic lives in `src/`
so it can be unit-tested and reused outside the notebook.


In [ ]:
import sys
sys.path.append("..")

from src.data import load_dataset
from src.solvers import SOLVERS
from src.experiment import run_experiments_per_class, build_summary_table
from src.anomaly import compute_anomaly_scores
from src.plotting import (
    plot_convergence_comparison, plot_dataset_convergence,
    combined_time_heatmap, report_extreme_samples,
)

DATASETS = ["mnist", "fashion_mnist", "cifar10"]


## 1. Run experiments (3 solvers x 3 datasets x 10 classes)

Each class gets an independent MEB per solver, trained under a shared 15s time budget and 1e-10 duality-gap tolerance.

In [ ]:
all_results = {}

for ds_name in DATASETS:
    print(f"\n{'=' * 100}\n  DATASET: {ds_name.upper()}\n{'=' * 100}")
    X_train, X_test, y_train, y_test = load_dataset(ds_name)
    models_by_solver = run_experiments_per_class(X_train, y_train, SOLVERS)
    all_results[ds_name] = {
        "models": models_by_solver,
        "X_test": X_test,
        "y_test": y_test,
    }
    del X_train, y_train


## 2. Convergence analysis

In [ ]:
for ds_name in DATASETS:
    plot_convergence_comparison(all_results[ds_name]["models"], dataset_name=ds_name.upper())


In [ ]:
for ds_name in DATASETS:
    plot_dataset_convergence(all_results, ds_name)


In [ ]:
combined_time_heatmap(all_results)


## 3. Summary table

Average time, iterations and final duality gap, computed only over the classes where each solver actually converged within the time budget.

In [ ]:
summary_df = build_summary_table(all_results)
summary_df


## 4. Anomaly detection

Each trained MEB acts as a compact model of \"what a normal member of that class looks like\". A test point's anomaly score is its distance from the ball's center, normalized by the ball's radius.

In [ ]:
for ds_name in DATASETS:
    X_test = all_results[ds_name]["X_test"]
    y_test = all_results[ds_name]["y_test"]

    for solver_name, models in all_results[ds_name]["models"].items():
        print(f"\n{'=' * 80}\n  ANOMALY DETECTION -- {ds_name.upper()} | {solver_name}\n{'=' * 80}")
        distances, ratios = compute_anomaly_scores(models, X_test, y_test)
        outside = (ratios > 1.0).sum()
        print(f"Samples outside the MEB (ratio > 1): {outside} / {len(ratios)}")
        report_extreme_samples(models, X_test, y_test, ratios,
                                dataset_name=ds_name.upper(), solver_name=solver_name, k=3)
